In [0]:
# ============================================
# NOTEBOOK 2: Silver Layer - Data Cleaning
# PROJECT: Wikipedia Clickstream Analytics
# ============================================

# Load bronze tables
bronze_clickstream = spark.table("workspace.default.bronze_clickstream")
bronze_pageviews = spark.table("workspace.default.bronze_pageviews")

print("Bronze Clickstream count:", bronze_clickstream.count())
print("Bronze Pageviews count:", bronze_pageviews.count())
print("✅ Bronze tables loaded successfully!")

Bronze Clickstream count: 34170973
Bronze Pageviews count: 4977578
✅ Bronze tables loaded successfully!


In [0]:
# ============================================
# SILVER: Clean Clickstream Data
# ============================================
from pyspark.sql.functions import col, current_timestamp

# 1. Remove bot traffic (other-empty, other-internal are bots/automated)
# 2. Keep only valid click types
# 3. Remove nulls
# 4. Remove zero or negative click counts

valid_click_types = ["link", "external", "other"]

silver_clickstream = bronze_clickstream \
    .filter(col("source_page").isNotNull()) \
    .filter(col("target_page").isNotNull()) \
    .filter(col("click_type").isin(valid_click_types)) \
    .filter(col("click_count") > 0) \
    .filter(~col("source_page").startswith("other-")) \
    .withColumn("ingested_at", current_timestamp())

print("Bronze clickstream count:", bronze_clickstream.count())
print("Silver clickstream count:", silver_clickstream.count())
print("Records removed:", bronze_clickstream.count() - silver_clickstream.count())
display(silver_clickstream.limit(5))

Bronze clickstream count: 34170973
Silver clickstream count: 24522123
Records removed: 9648850


source_page,target_page,click_type,click_count,ingested_at
Chinese_water_dragon,Arthropod,link,12,2026-04-24T05:55:21.059Z
Chinese_water_snake,Snake,link,12,2026-04-24T05:55:21.059Z
Chinese_water_torture,Waterboarding,other,12,2026-04-24T05:55:21.059Z
Chinese_zodiac,Shang_dynasty,link,12,2026-04-24T05:55:21.059Z
Chinese_zodiac,Wild_boar,link,12,2026-04-24T05:55:21.059Z


In [0]:
# ============================================
# SILVER: Clean Pageview Data
# ============================================
from pyspark.sql.functions import col, current_timestamp, regexp_replace

# 1. Keep only English Wikipedia (language_code = 'en')
# 2. Remove special pages (File:, Module:, Special:, Talk:, User:)
# 3. Remove null article titles
# 4. Remove zero view counts

silver_pageviews = bronze_pageviews \
    .filter(col("language_code") == "en") \
    .filter(col("article_title").isNotNull()) \
    .filter(col("view_count") > 0) \
    .filter(~col("article_title").startswith("File:")) \
    .filter(~col("article_title").startswith("Special:")) \
    .filter(~col("article_title").startswith("Module:")) \
    .filter(~col("article_title").startswith("Talk:")) \
    .filter(~col("article_title").startswith("User:")) \
    .withColumn("ingested_at", current_timestamp())

print("Bronze pageviews count:", bronze_pageviews.count())
print("Silver pageviews count:", silver_pageviews.count())
print("Records removed:", bronze_pageviews.count() - silver_pageviews.count())
display(silver_pageviews.limit(5))

Bronze pageviews count: 4977578
Silver pageviews count: 861793
Records removed: 4115785


language_code,article_title,view_count,response_size,ingested_at
en,!Bang!,1,0,2026-04-24T05:57:13.439Z
en,!Kung_languages,1,0,2026-04-24T05:57:13.439Z
en,!Kung_mythology,1,0,2026-04-24T05:57:13.439Z
en,!Lander,1,0,2026-04-24T05:57:13.439Z
en,!_(Trippie_Redd_album),2,0,2026-04-24T05:57:13.439Z


In [0]:
# ============================================
# SILVER: Save Cleaned Data to Delta Tables
# ============================================

# Save silver clickstream
silver_clickstream.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_clickstream")

print("✅ silver_clickstream table saved!")

# Save silver pageviews
silver_pageviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_pageviews")

print("✅ silver_pageviews table saved!")
print("🎉 Silver Layer Complete!")

✅ silver_clickstream table saved!
✅ silver_pageviews table saved!
🎉 Silver Layer Complete!
